In [ ]:
import sys
import os
from langchain_core.documents import Document

# 1. 경로 설정 및 모듈 임포트
PROJECT_ROOT = "/home/taehoon/sprint-ai-mid-project_team3"
if os.path.abspath(PROJECT_ROOT) not in sys.path:
    sys.path.append(os.path.abspath(PROJECT_ROOT))

from src.retriever_factory import RetrieverFactory

# 2. 팩토리를 통해 local 프로필로 Retriever 생성
retriever = RetrieverFactory.create_retriever(profile="local", config_path="../config/default.yaml")

# 3. 지정된 경로에서 sample_rfp.txt 파일을 읽어와 청킹 및 메타데이터 동적 부여
FILE_PATH = os.path.join(PROJECT_ROOT, "samples/raw/sample_rfp.txt")

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(f"지정한 경로에 파일이 존재하지 않습니다: {FILE_PATH}")

with open(FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

# [전처리] 텍스트를 문단(연속된 두 줄 줄바꿈) 단위로 분할하여 청크 생성
paragraphs = [p.strip() for p in raw_text.split("\n\n") if p.strip()]
processed_docs = []

for idx, paragraph in enumerate(paragraphs):
    if idx % 2 == 0:
        org = "과학기술정보통신부"
        project = "지능형_AI_플랫폼_구축"
    else:
        org = "산업통상자원부"
        project = "에너지_빅데이터_분석"
        
    doc = Document(
        page_content=paragraph,
        metadata={
            "org_name": org,
            "project_name": project,
            "doc_id": f"DOC_RFP_{idx:03d}"
        }
    )
    processed_docs.append(doc)

print(f"총 {len(processed_docs)}개의 텍스트 문서 청크가 준비되었습니다.")
retriever.add_documents(processed_docs)
print("Chroma DB에 sample_rfp 데이터 적재 완료!\n")


# 4. 메타데이터 필터링 통합 검색 검증
query_str = "사업의 추진 배경 및 기대 효과를 알려주세요"

print("=== 1. 필터 없이 유사도 전체 검색 (Top-3) ===")
results = retriever.retrieve(query=query_str, top_k=3)
for d in results:
    print(f"[{d.metadata['doc_id']} / {d.metadata['org_name']}] {d.page_content[:60]}...")

print("\n=== 2. 기관명 필터 적용 (과학기술정보통신부 데이터만 필터링) ===")
results_filtered = retriever.retrieve(query=query_str, org_name="과학기술정보통신부", top_k=2)
for d in results_filtered:
    print(f"[{d.metadata['org_name']} / {d.metadata['project_name']}] {d.page_content[:60]}...")

print("\n=== 3. 특정 문서 고유 ID 정밀 검색 (DOC_RFP_001 강제 지정) ===")
results_doc = retriever.retrieve(query=query_str, doc_id="DOC_RFP_001")
if results_doc:
    print(f"[{results_doc[0].metadata['doc_id']}] {results_doc[0].page_content}")


print("\n" + "="*50)
print("=== 5. (추가) Multi-Collection 방식을 적용한 하이브리드 컬렉션 검증 ===")
print("="*50)

# 1. 과기부 전용 리트리버 생성 및 데이터 적재
msit_retriever = RetrieverFactory.create_retriever(profile="local", org_name="과학기술정보통신부")

# 2. 산자부 전용 리트리버 생성 및 데이터 적재
motie_retriever = RetrieverFactory.create_retriever(profile="local", org_name="산업통상자원부")

# 3. 검색 시에는 해당 기관 리트리버에서 사업명(project_name) 필터만 걸고 검색
results_hybrid = msit_retriever.retrieve(query="보안 기술", project_name="AI보안사업")
print("Multi-Collection 하이브리드 테스트 호출 완료")